In [ ]:
#!/usr/bin/env python3
"""
Spin-resolved pd-hybridization fatband plotter.

Usage:
    python plot_fatband.py

Configuration:
    Edit the CONFIG section below to match your data.
"""

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable
from matplotlib.ticker import MultipleLocator
plt.rcParams['mathtext.fontset'] = 'cm'
plt.rcParams['font.family'] = 'Times New Roman'


# ============================================================================
# 함수 정의 — 수정 불필요
# ============================================================================

def read_pband_data(filename):
    """PBAND_SUM 파일을 읽어 밴드별 numpy 배열 리스트로 반환"""
    bands = []
    current_band = []
    with open(filename, 'r') as f:
        for line in f:
            if line.startswith('#'):
                continue
            if line.strip() == '':
                if current_band:
                    bands.append(np.array(current_band))
                    current_band = []
            else:
                data = list(map(float, line.split()))
                current_band.append(data)
    if current_band:
        bands.append(np.array(current_band))
    return bands


def calculate_hyb(s_occ, p_occ, d_occ):
    """
    pd-hybridization index 계산.
    
    H = 2 * min(p_frac, d_frac)
      - p_frac = p / (s + p + d)
      - d_frac = d / (s + p + d)
    
    p=d=0.5 일 때 H=1.0 (최대), 한쪽 지배적이면 H→0.
    """
    size = np.zeros(len(p_occ))
    for i in range(len(p_occ)):
        total = s_occ[i] + p_occ[i] + d_occ[i]
        if total > 1e-6:
            size[i] = 2 * min(p_occ[i] / total, d_occ[i] / total)
    return size


def extract_orbital_weights(band):
    """밴드 데이터에서 s, p, d occupation을 추출"""
    s_occ = band[:, 2]
    p_occ = band[:, 3] + band[:, 4] + band[:, 5]          # py + pz + px
    d_occ = band[:, 6] + band[:, 7] + band[:, 8] + band[:, 9] + band[:, 10]  # dxy+dyz+dz2+dxz+dx2-y2
    return s_occ, p_occ, d_occ


def find_sse_kpoint(bands_up, bands_dw, sse_band_0, sse_e_up=None, sse_e_dw=None, sse_kpt_1based=None):
    """
    SSE 밴드에서 SSE가 최대인 k-point index (0-based) 찾기.
    
    방법 1: sse_kpt_1based가 주어지면, PBAND 데이터에서 에너지가 가장 가까운 점을 찾음.
    방법 2: 단순히 spin splitting이 최대인 점을 찾음.
    """
    band_up = bands_up[sse_band_0]
    band_dw = bands_dw[sse_band_0]
    
    # 방법: splitting 최대 지점
    splitting = np.abs(band_up[:, 1] - band_dw[:, 1])
    return np.argmax(splitting)


def plot_fatband(file_up, file_dw, sse_band_1based, sse_value,
                 klabels, kpositions='auto', energy_range=(-4, 4),
                 s_normal=20, s_sse=80, show_colorbar=True,
                 save_path=None, dpi=300):
    """
    Spin-resolved pd-hybridization fatband 플롯.
    
    Parameters
    ----------
    file_up, file_dw : str
        PBAND_SUM_UP.dat / DW.dat 경로
    sse_band_1based : int
        SSE 밴드 인덱스 (1-based, SSE 스크립트 출력 그대로)
    sse_value : float
        SSE 값 (eV), 표시용
    klabels : list of str
        고대칭점 라벨 리스트
    kpositions : 'auto' or list of float
        라벨 위치. 'auto'이면 균등 배치.
    energy_range : tuple
        (E_min, E_max) in eV
    s_normal, s_sse : float
        일반/SSE 밴드 마커 크기
    show_colorbar : bool
        True이면 spin-up/down 컬러바 표시, False이면 숨김
    save_path : str or None
        저장 경로
    dpi : int
        저장 해상도
        
    Returns
    -------
    fig, ax
    """
    # ── 데이터 읽기 ──
    bands_up = read_pband_data(file_up)
    bands_dw = read_pband_data(file_dw)
    
    n_bands = len(bands_up)
    n_kpts = len(bands_up[0])
    max_kpath = max(band[:, 0].max() for band in bands_up)
    
    sse_band_0 = sse_band_1based - 1  # 0-based 변환
    
    print(f"Bands: {n_bands}, K-points: {n_kpts}")
    print(f"K-path range: 0 ~ {max_kpath:.5f}")
    print(f"SSE band: {sse_band_1based} (1-based) = {sse_band_0} (0-based)")
    
    fig, ax = plt.subplots(figsize=(6.4, 6.4))
    
    scatter_up = scatter_dw = None
    
    # ── Spin-up 밴드 ──
    for idx, band in enumerate(bands_up):
        kpath = band[:, 0]
        energy = band[:, 1]
        s_occ, p_occ, d_occ = extract_orbital_weights(band)
        cv = calculate_hyb(s_occ, p_occ, d_occ)
        mask = cv > 0
        
        is_sse = (idx == sse_band_0)
        ms = s_sse if is_sse else s_normal
        zord = 5 if is_sse else 2
        ew = 0.3 if is_sse else 0
        ec = 'darkred' if is_sse else 'none'
        
        if np.any(mask):
            sc = ax.scatter(kpath[mask], energy[mask], s=ms,
                     c=cv[mask], cmap='Reds', vmin=0., vmax=1.0,
                     alpha=1, edgecolors=ec, linewidths=ew, zorder=zord)
            scatter_up = sc
    
    # ── Spin-down 밴드 ──
    for idx, band in enumerate(bands_dw):
        kpath = band[:, 0]
        energy = band[:, 1]
        s_occ, p_occ, d_occ = extract_orbital_weights(band)
        cv = calculate_hyb(s_occ, p_occ, d_occ)
        mask = cv > 0
        
        is_sse = (idx == sse_band_0)
        ms = s_sse if is_sse else s_normal
        zord = 5 if is_sse else 2
        ew = 0.3 if is_sse else 0
        ec = 'darkblue' if is_sse else 'none'
        
        if np.any(mask):
            sc = ax.scatter(kpath[mask], energy[mask], s=ms,
                     c=cv[mask], cmap='Blues', vmin=0, vmax=1.0,
                     alpha=1, edgecolors=ec, linewidths=ew, zorder=zord)
            scatter_dw = sc
    
    # ── 축 설정 ──
    y_min, y_max = energy_range

    ax.set_ylabel(r'$\mathrm{E - E_F\ (eV)}$', fontsize=25)
    ax.set_xlim(0, max_kpath)
    ax.set_ylim(energy_range)

    ax.set_yticks(np.arange(y_min, y_max + 1, 1))
    ax.tick_params(axis='both', which='major', labelsize=25)
    
    # K-path 라벨
    if kpositions == 'auto':
        n_labels = len(klabels)
        kpos = np.linspace(0, max_kpath, n_labels)
    else:
        kpos = kpositions
    
    ax.set_xticks(kpos)
    ax.set_xticklabels(klabels, fontsize=25)
    
    # 고대칭점 세로선 (양 끝 제외)
    for kp in kpos[1:-1]:
        ax.axvline(x=kp, color='black', lw=1.0, linestyle='--', dashes=(4, 3), zorder=1)
    
    # Fermi level
    ax.axhline(y=0, color='black', ls='--', lw=1.2, zorder=1)
    
    # ── 컬러바 ──
    if show_colorbar:
        divider = make_axes_locatable(ax)
        cax1 = divider.append_axes("right", size="4%", pad=0.05)
        cax2 = divider.append_axes("right", size="4%", pad=0.9)
        
        if scatter_up is not None:
            cbar1 = plt.colorbar(scatter_up, cax=cax1)
            cbar1.set_label('spin up', fontsize=25, rotation=90, labelpad=1)
            cbar1.ax.tick_params(labelsize=25)
        
        if scatter_dw is not None:
            cbar2 = plt.colorbar(scatter_dw, cax=cax2)
            cbar2.set_label('spin down', fontsize=25, rotation=90, labelpad=1)
            cbar2.ax.tick_params(labelsize=25)
    
    plt.tight_layout()
    
    # ── 저장 ──
    if save_path:
        plt.savefig(save_path, dpi=dpi, bbox_inches='tight')
        print(f"Saved: {save_path}")
    
    return fig, ax

# ============================================================================
# CONFIG — 여기만 수정하세요
# ============================================================================

# 1. 파일 경로
FILE_UP = 'CrSb/PBAND_SUM_UP.dat'
FILE_DW = 'CrSb/PBAND_SUM_DW.dat'

# 2. SSE 정보 (SSE 스크립트 출력에서 가져오세요)
#    Band Index는 1-based (SSE 출력 그대로), 코드가 자동으로 0-based 변환
SSE_BAND_INDEX = 17       # Band Index from SSE output
SSE_KPOINT_INDEX = 24     # K-Point Index from SSE output (1-based)
SSE_VALUE = 1.27          # Max Energy Difference (eV)

# 3. K-path 라벨 설정
KLABELS = [r"L", r"$\Gamma$", r"L'"]
KPOSITIONS = 'auto'  # 'auto' or list of kpath values

# 4. 에너지 범위
ENERGY_RANGE = (-4, 4)

# 5. 마커 크기
S_NORMAL = 20    # 일반 밴드 마커 크기
S_SSE = 80       # SSE 밴드 마커 크기

# 6. 컬러바 표시 여부
SHOW_COLORBAR = True   # False로 바꾸면 컬러바 숨김

# 7. 저장 설정
SAVE_PATH = 'CrSb_fatband.png'  # None이면 저장 안 함
DPI = 300

# ============================================================================
# 실행
# ============================================================================
if __name__ == '__main__':
    fig, ax = plot_fatband(
        file_up=FILE_UP,
        file_dw=FILE_DW,
        sse_band_1based=SSE_BAND_INDEX,
        sse_value=SSE_VALUE,
        klabels=KLABELS,
        kpositions=KPOSITIONS,
        energy_range=ENERGY_RANGE,
        s_normal=S_NORMAL,
        s_sse=S_SSE,
        show_colorbar=SHOW_COLORBAR,
        save_path=SAVE_PATH,
        dpi=DPI,
    )
    plt.show()

In [ ]:
# ============================================================================
# CONFIG — 여기만 수정하세요
# ============================================================================

# 1. 파일 경로
FILE_UP = 'VO/PBAND_SUM_UP.dat'
FILE_DW = 'VO/PBAND_SUM_DW.dat'

# 2. SSE 정보 (SSE 스크립트 출력에서 가져오세요)
#    Band Index는 1-based (SSE 출력 그대로), 코드가 자동으로 0-based 변환
SSE_BAND_INDEX = 18       # Band Index from SSE output
SSE_KPOINT_INDEX = 60     # K-Point Index from SSE output (1-based)
SSE_VALUE = 0.179          # Max Energy Difference (eV)

# 3. K-path 라벨 설정
#    k-point 경로의 고대칭점 라벨
#    labels: 표시할 라벨 리스트
#    positions: 'auto' 이면 [시작, 중간, 끝] 자동 배치
#               직접 kpath 값 리스트로 지정 가능 (e.g., [0, 0.5, 1.0, 1.5])
KLABELS = [r"L", r"$\Gamma$", r"L'"]
KPOSITIONS = 'auto'  # 'auto' or list of kpath values

# 4. 에너지 범위
ENERGY_RANGE = (-4, 4)

# 5. 마커 크기
S_NORMAL = 20    # 일반 밴드 마커 크기
S_SSE = 80       # SSE 밴드 마커 크기

# 6. 저장 설정
SAVE_PATH = 'VO_fatband.png'  # None이면 저장 안 함
DPI = 300

# ============================================================================
# 실행
# ============================================================================
if __name__ == '__main__':
    fig, ax = plot_fatband(
        file_up=FILE_UP,
        file_dw=FILE_DW,
        sse_band_1based=SSE_BAND_INDEX,
        sse_value=SSE_VALUE,
        klabels=KLABELS,
        kpositions=KPOSITIONS,
        energy_range=ENERGY_RANGE,
        s_normal=S_NORMAL,
        s_sse=S_SSE,
        show_colorbar=False,
        save_path=SAVE_PATH,
        dpi=DPI,
    )
    plt.show()

In [ ]:
#!/usr/bin/env python3
"""
Spin-resolved pd-hybridization fatband plotter — combined two-panel figure.

Left panel  : VO   (no colorbar, with y-label)
Right panel : CrSb (with colorbar, no y-label)

Band-area sizes are kept identical by always reserving colorbar space
and toggling visibility.

Usage:
    python plot_fatband_combined.py
"""

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable
plt.rcParams['mathtext.fontset'] = 'cm'
plt.rcParams['font.family'] = 'Times New Roman'


# ============================================================================
# CONFIG — 여기만 수정하세요
# ============================================================================

PANELS = [
    # ── 왼쪽 패널: VO ──────────────────────────────────────────────────────
    dict(
        file_up        = 'VO/PBAND_SUM_UP.dat',
        file_dw        = 'VO/PBAND_SUM_DW.dat',
        sse_band_index = 18,       # 1-based
        sse_kpt_index  = 60,       # 1-based (참고용)
        sse_value      = 0.179,    # eV
        klabels        = [r'L', r'$\Gamma$', r"L'"],
        kpositions     = 'auto',
        energy_range   = (-4, 4),
        s_normal       = 20,
        s_sse          = 80,
        show_colorbar  = False,    # 왼쪽: colorbar 없음
        show_ylabel    = True,     # 왼쪽: y-label 있음
    ),
    # ── 오른쪽 패널: CrSb ──────────────────────────────────────────────────
    dict(
        file_up        = 'CrSb/PBAND_SUM_UP.dat',
        file_dw        = 'CrSb/PBAND_SUM_DW.dat',
        sse_band_index = 17,
        sse_kpt_index  = 24,
        sse_value      = 1.27,
        klabels        = [r'L', r'$\Gamma$', r"L'"],
        kpositions     = 'auto',
        energy_range   = (-4, 4),
        s_normal       = 20,
        s_sse          = 80,
        show_colorbar  = True,     # 오른쪽: colorbar 있음
        show_ylabel    = False,    # 오른쪽: y-label 없음
    ),
]

# 저장 설정
SAVE_PATH = 'fatband_combined.png'   # None이면 저장 안 함
DPI       = 300

# 패널 크기 (인치)
PANEL_W, PANEL_H = 6.4, 6.4


# ============================================================================
# 함수 정의 — 수정 불필요
# ============================================================================

def read_pband_data(filename):
    bands, current_band = [], []
    with open(filename, 'r') as f:
        for line in f:
            if line.startswith('#'):
                continue
            if line.strip() == '':
                if current_band:
                    bands.append(np.array(current_band))
                    current_band = []
            else:
                current_band.append(list(map(float, line.split())))
    if current_band:
        bands.append(np.array(current_band))
    return bands


def calculate_hyb(s_occ, p_occ, d_occ):
    size = np.zeros(len(p_occ))
    for i in range(len(p_occ)):
        total = s_occ[i] + p_occ[i] + d_occ[i]
        if total > 1e-6:
            size[i] = 2 * min(p_occ[i] / total, d_occ[i] / total)
    return size


def extract_orbital_weights(band):
    s_occ = band[:, 2]
    p_occ = band[:, 3] + band[:, 4] + band[:, 5]
    d_occ = band[:, 6] + band[:, 7] + band[:, 8] + band[:, 9] + band[:, 10]
    return s_occ, p_occ, d_occ


def draw_panel(ax, file_up, file_dw, sse_band_index, sse_value,
               klabels, kpositions, energy_range,
               s_normal, s_sse, show_colorbar, show_ylabel):
    """
    밴드 하나를 ax에 그림.
    colorbar 공간은 show_colorbar 여부에 관계없이 항상 예약하여
    밴드 영역 크기가 두 패널에서 동일하게 유지됨.
    """
    bands_up = read_pband_data(file_up)
    bands_dw = read_pband_data(file_dw)

    n_bands  = len(bands_up)
    n_kpts   = len(bands_up[0])
    max_kpath = max(b[:, 0].max() for b in bands_up)
    sse_band_0 = sse_band_index - 1   # 0-based

    print(f"  Bands: {n_bands}, K-points: {n_kpts}, "
          f"SSE band: {sse_band_index} (1-based)")

    scatter_up = scatter_dw = None

    # ── Spin-up ──
    for idx, band in enumerate(bands_up):
        kpath  = band[:, 0]
        energy = band[:, 1]
        s_occ, p_occ, d_occ = extract_orbital_weights(band)
        cv   = calculate_hyb(s_occ, p_occ, d_occ)
        mask = cv > 0
        is_sse = (idx == sse_band_0)
        sc = ax.scatter(
            kpath[mask], energy[mask],
            s          = s_sse if is_sse else s_normal,
            c          = cv[mask],
            cmap       = 'Reds', vmin=0., vmax=1.0,
            alpha      = 1,
            edgecolors = 'darkred' if is_sse else 'none',
            linewidths = 0.3 if is_sse else 0,
            zorder     = 5 if is_sse else 2,
        ) if np.any(mask) else None
        if sc is not None:
            scatter_up = sc

    # ── Spin-down ──
    for idx, band in enumerate(bands_dw):
        kpath  = band[:, 0]
        energy = band[:, 1]
        s_occ, p_occ, d_occ = extract_orbital_weights(band)
        cv   = calculate_hyb(s_occ, p_occ, d_occ)
        mask = cv > 0
        is_sse = (idx == sse_band_0)
        sc = ax.scatter(
            kpath[mask], energy[mask],
            s          = s_sse if is_sse else s_normal,
            c          = cv[mask],
            cmap       = 'Blues', vmin=0., vmax=1.0,
            alpha      = 1,
            edgecolors = 'darkblue' if is_sse else 'none',
            linewidths = 0.3 if is_sse else 0,
            zorder     = 5 if is_sse else 2,
        ) if np.any(mask) else None
        if sc is not None:
            scatter_dw = sc

    # ── 축 ──
    y_min, y_max = energy_range
    ax.set_xlim(0, max_kpath)
    ax.set_ylim(energy_range)
    ax.set_yticks(np.arange(y_min, y_max + 1, 1))
    ax.tick_params(axis='both', which='major', labelsize=25)

    if show_ylabel:
        ax.set_ylabel(r'$\mathrm{E - E_F\ (eV)}$', fontsize=25)
    # ytick 숫자는 show_ylabel 여부와 무관하게 항상 표시

    # K-path 라벨
    kpos = (np.linspace(0, max_kpath, len(klabels))
            if kpositions == 'auto' else np.asarray(kpositions))
    ax.set_xticks(kpos)
    ax.set_xticklabels(klabels, fontsize=25)

    for kp in kpos[1:-1]:
        ax.axvline(x=kp, color='black', lw=1.0, linestyle='--', dashes=(4, 3), zorder=1)
    ax.axhline(y=0, color='black', ls='--', lw=1.2, zorder=1)

    # ── 컬러바 공간 — 항상 예약, visible 여부만 다름 ──
    # → 두 패널의 밴드 영역 크기가 동일하게 유지됨
    divider = make_axes_locatable(ax)
    cax1 = divider.append_axes("right", size="4%", pad=0.05)
    cax2 = divider.append_axes("right", size="4%", pad=0.9)

    if show_colorbar:
        if scatter_up is not None:
            cbar1 = plt.colorbar(scatter_up, cax=cax1)
            cbar1.set_label('spin-up',   fontsize=25, rotation=90, labelpad=1)
            cbar1.ax.tick_params(labelsize=20)
        if scatter_dw is not None:
            cbar2 = plt.colorbar(scatter_dw, cax=cax2)
            cbar2.set_label('spin-down', fontsize=25, rotation=90, labelpad=1)
            cbar2.ax.tick_params(labelsize=20)
    else:
        cax1.set_visible(False)
        cax2.set_visible(False)


# ============================================================================
# 실행
# ============================================================================

if __name__ == '__main__':
    n = len(PANELS)
    fig, axes = plt.subplots(1, n, figsize=(5.5 * n, 6))
    if n == 1:
        axes = [axes]

    for ax, cfg in zip(axes, PANELS):
        print(f"\nDrawing: {cfg['file_up']}")
        draw_panel(
            ax            = ax,
            file_up       = cfg['file_up'],
            file_dw       = cfg['file_dw'],
            sse_band_index= cfg['sse_band_index'],
            sse_value     = cfg['sse_value'],
            klabels       = cfg['klabels'],
            kpositions    = cfg['kpositions'],
            energy_range  = cfg['energy_range'],
            s_normal      = cfg['s_normal'],
            s_sse         = cfg['s_sse'],
            show_colorbar = cfg['show_colorbar'],
            show_ylabel   = cfg['show_ylabel'],
        )

    plt.tight_layout()

    if SAVE_PATH:
        plt.savefig(SAVE_PATH, dpi=DPI, bbox_inches='tight')
        print(f"\nSaved: {SAVE_PATH}")

    plt.show()

In [ ]:
'../final_structure/NiS/PBAND_SUM_UP.dat'

In [ ]:
#!/usr/bin/env python3
"""
Spin-resolved pd-hybridization fatband plotter — combined two-panel figure.

Left panel  : VO   (no colorbar, with y-label)
Right panel : CrSb (with colorbar, no y-label)

Band-area sizes are kept identical by always reserving colorbar space
and toggling visibility.

Usage:
    python plot_fatband_combined.py
"""

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable
plt.rcParams['mathtext.fontset'] = 'cm'
plt.rcParams['font.family'] = 'Times New Roman'


# ============================================================================
# CONFIG — 여기만 수정하세요
# ============================================================================

PANELS = [
    # ── 왼쪽 패널: VO ──────────────────────────────────────────────────────
    dict(
        file_up        = '../final_structure/FeS/PBAND_SUM_UP.dat',
        file_dw        = '../final_structure/FeS/PBAND_SUM_DW.dat',
        sse_band_index = 14,       # 1-based
        sse_kpt_index  = 66,       # 1-based (참고용)
        sse_value      = 1.296646,    # eV
        klabels        = [r'T$_2$', r'$\Gamma$', r"U$_2$"],
        kpositions     = [0,1.289,2.579],
        energy_range   = (-4, 4),
        s_normal       = 20,
        s_sse          = 80,
        show_colorbar  = False,    # 왼쪽: colorbar 없음
        show_ylabel    = True,     # 왼쪽: y-label 있음
    ),
    # ── 오른쪽 패널: CrSb ──────────────────────────────────────────────────
    dict(
        file_up        = '../final_structure/NiS/PBAND_SUM_UP.dat',
        file_dw        = '../final_structure/NiS/PBAND_SUM_DW.dat',
        sse_band_index = 16,
        sse_kpt_index  = 63,
        sse_value      = 0.822971,
        klabels        = [r'L', r'$\Gamma$', r"L'"],
        kpositions     = 'auto',
        energy_range   = (-4, 4),
        s_normal       = 20,
        s_sse          = 80,
        show_colorbar  = True,     # 오른쪽: colorbar 있음
        show_ylabel    = False,    # 오른쪽: y-label 없음
    ),
]

# 저장 설정
SAVE_PATH = 'fatband_combined_FeS_NiS.png'   # None이면 저장 안 함
DPI       = 300

# 패널 크기 (인치)
PANEL_W, PANEL_H = 6.4, 6.4


# ============================================================================
# 함수 정의 — 수정 불필요
# ============================================================================

def read_pband_data(filename):
    bands, current_band = [], []
    with open(filename, 'r') as f:
        for line in f:
            if line.startswith('#'):
                continue
            if line.strip() == '':
                if current_band:
                    bands.append(np.array(current_band))
                    current_band = []
            else:
                current_band.append(list(map(float, line.split())))
    if current_band:
        bands.append(np.array(current_band))
    return bands


def calculate_hyb(s_occ, p_occ, d_occ):
    size = np.zeros(len(p_occ))
    for i in range(len(p_occ)):
        total = s_occ[i] + p_occ[i] + d_occ[i]
        if total > 1e-6:
            size[i] = 2 * min(p_occ[i] / total, d_occ[i] / total)
    return size


def extract_orbital_weights(band):
    s_occ = band[:, 2]
    p_occ = band[:, 3] + band[:, 4] + band[:, 5]
    d_occ = band[:, 6] + band[:, 7] + band[:, 8] + band[:, 9] + band[:, 10]
    return s_occ, p_occ, d_occ


def draw_panel(ax, file_up, file_dw, sse_band_index, sse_value,
               klabels, kpositions, energy_range,
               s_normal, s_sse, show_colorbar, show_ylabel):
    """
    밴드 하나를 ax에 그림.
    colorbar 공간은 show_colorbar 여부에 관계없이 항상 예약하여
    밴드 영역 크기가 두 패널에서 동일하게 유지됨.
    """
    bands_up = read_pband_data(file_up)
    bands_dw = read_pband_data(file_dw)

    n_bands  = len(bands_up)
    n_kpts   = len(bands_up[0])
    max_kpath = max(b[:, 0].max() for b in bands_up)
    sse_band_0 = sse_band_index - 1   # 0-based

    print(f"  Bands: {n_bands}, K-points: {n_kpts}, "
          f"SSE band: {sse_band_index} (1-based)")

    scatter_up = scatter_dw = None

    # ── Spin-up ──
    for idx, band in enumerate(bands_up):
        kpath  = band[:, 0]
        energy = band[:, 1]
        s_occ, p_occ, d_occ = extract_orbital_weights(band)
        cv   = calculate_hyb(s_occ, p_occ, d_occ)
        mask = cv > 0
        is_sse = (idx == sse_band_0)
        sc = ax.scatter(
            kpath[mask], energy[mask],
            s          = s_sse if is_sse else s_normal,
            c          = cv[mask],
            cmap       = 'Reds', vmin=0., vmax=1.0,
            alpha      = 1,
            edgecolors = 'darkred' if is_sse else 'none',
            linewidths = 0.3 if is_sse else 0,
            zorder     = 5 if is_sse else 2,
        ) if np.any(mask) else None
        if sc is not None:
            scatter_up = sc

    # ── Spin-down ──
    for idx, band in enumerate(bands_dw):
        kpath  = band[:, 0]
        energy = band[:, 1]
        s_occ, p_occ, d_occ = extract_orbital_weights(band)
        cv   = calculate_hyb(s_occ, p_occ, d_occ)
        mask = cv > 0
        is_sse = (idx == sse_band_0)
        sc = ax.scatter(
            kpath[mask], energy[mask],
            s          = s_sse if is_sse else s_normal,
            c          = cv[mask],
            cmap       = 'Blues', vmin=0., vmax=1.0,
            alpha      = 1,
            edgecolors = 'darkblue' if is_sse else 'none',
            linewidths = 0.3 if is_sse else 0,
            zorder     = 5 if is_sse else 2,
        ) if np.any(mask) else None
        if sc is not None:
            scatter_dw = sc

    # ── 축 ──
    y_min, y_max = energy_range
    ax.set_xlim(0, max_kpath)
    ax.set_ylim(energy_range)
    ax.set_yticks(np.arange(y_min, y_max + 1, 1))
    ax.tick_params(axis='both', which='major', labelsize=25)

    if show_ylabel:
        ax.set_ylabel(r'$\mathrm{E - E_F\ (eV)}$', fontsize=25)
    # ytick 숫자는 show_ylabel 여부와 무관하게 항상 표시

    # K-path 라벨
    kpos = (np.linspace(0, max_kpath, len(klabels))
            if kpositions == 'auto' else np.asarray(kpositions))
    ax.set_xticks(kpos)
    ax.set_xticklabels(klabels, fontsize=25)

    for kp in kpos[1:-1]:
        ax.axvline(x=kp, color='black', lw=1.0, linestyle='--', dashes=(4, 3), zorder=1)
    ax.axhline(y=0, color='black', ls='--', lw=1.2, zorder=1)

    # ── 컬러바 공간 — 항상 예약, visible 여부만 다름 ──
    # → 두 패널의 밴드 영역 크기가 동일하게 유지됨
    divider = make_axes_locatable(ax)
    cax1 = divider.append_axes("right", size="4%", pad=0.05)
    cax2 = divider.append_axes("right", size="4%", pad=0.9)

    if show_colorbar:
        if scatter_up is not None:
            cbar1 = plt.colorbar(scatter_up, cax=cax1)
            cbar1.set_label('spin-up',   fontsize=25, rotation=90, labelpad=1)
            cbar1.ax.tick_params(labelsize=20)
        if scatter_dw is not None:
            cbar2 = plt.colorbar(scatter_dw, cax=cax2)
            cbar2.set_label('spin-down', fontsize=25, rotation=90, labelpad=1)
            cbar2.ax.tick_params(labelsize=20)
    else:
        cax1.set_visible(False)
        cax2.set_visible(False)


# ============================================================================
# 실행
# ============================================================================

if __name__ == '__main__':
    n = len(PANELS)
    fig, axes = plt.subplots(1, n, figsize=(5.5 * n, 6))
    if n == 1:
        axes = [axes]

    for ax, cfg in zip(axes, PANELS):
        print(f"\nDrawing: {cfg['file_up']}")
        draw_panel(
            ax            = ax,
            file_up       = cfg['file_up'],
            file_dw       = cfg['file_dw'],
            sse_band_index= cfg['sse_band_index'],
            sse_value     = cfg['sse_value'],
            klabels       = cfg['klabels'],
            kpositions    = cfg['kpositions'],
            energy_range  = cfg['energy_range'],
            s_normal      = cfg['s_normal'],
            s_sse         = cfg['s_sse'],
            show_colorbar = cfg['show_colorbar'],
            show_ylabel   = cfg['show_ylabel'],
        )

    plt.tight_layout()

    if SAVE_PATH:
        plt.savefig(SAVE_PATH, dpi=DPI, bbox_inches='tight')
        print(f"\nSaved: {SAVE_PATH}")

    plt.show()

SI

In [ ]:
#!/usr/bin/env python3
"""
Spin-resolved pd-hybridization fatband plotter — combined two-panel figure.

Left panel  : VO   (no colorbar, with y-label)
Right panel : CrSb (with colorbar, no y-label)

Band-area sizes are kept identical by always reserving colorbar space
and toggling visibility.

Usage:
    python plot_fatband_combined.py
"""

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable
plt.rcParams['mathtext.fontset'] = 'cm'
plt.rcParams['font.family'] = 'Times New Roman'


# ============================================================================
# CONFIG — 여기만 수정하세요
# ============================================================================

PANELS = [
    # ── 왼쪽 패널: VO ──────────────────────────────────────────────────────
    dict(
        file_up        = '../final_structure/CoO/PBAND_SUM_UP.dat',
        file_dw        = '../final_structure/CoO/PBAND_SUM_DW.dat',
        sse_band_index = 15,       # 1-based
        sse_kpt_index  = 1,       # 1-based (참고용)
        sse_value      = 0.972368,    # eV
        klabels        = [r'T$_2$', r'$\Gamma$', r"U"],
        kpositions     = [0, 1.457, 2.916],
        energy_range   = (-4, 4),
        s_normal       = 20,
        s_sse          = 80,
        show_colorbar  = False,    # 왼쪽: colorbar 없음
        show_ylabel    = True,     # 왼쪽: y-label 있음
    ),
    # ── 오른쪽 패널: CrSb ──────────────────────────────────────────────────
    dict(
        file_up        = '../final_structure/CrS/PBAND_SUM_UP.dat',
        file_dw        = '../final_structure/CrS/PBAND_SUM_DW.dat',
        sse_band_index = 15,
        sse_kpt_index  = 80,
        sse_value      = 0.314205,
        klabels        = [r'T$_2$', r'$\Gamma$', r"U"],
        kpositions     = [0, 1.151, 2.303],
        energy_range   = (-4, 4),
        s_normal       = 20,
        s_sse          = 80,
        show_colorbar  = True,     # 오른쪽: colorbar 있음
        show_ylabel    = False,    # 오른쪽: y-label 없음
    ),
]

# 저장 설정
SAVE_PATH = 'SI_fatband_combined_CoO_CrS.png'   # None이면 저장 안 함
DPI       = 300

# 패널 크기 (인치)
PANEL_W, PANEL_H = 6.4, 6.4


# ============================================================================
# 함수 정의 — 수정 불필요
# ============================================================================

def read_pband_data(filename):
    bands, current_band = [], []
    with open(filename, 'r') as f:
        for line in f:
            if line.startswith('#'):
                continue
            if line.strip() == '':
                if current_band:
                    bands.append(np.array(current_band))
                    current_band = []
            else:
                current_band.append(list(map(float, line.split())))
    if current_band:
        bands.append(np.array(current_band))
    return bands


def calculate_hyb(s_occ, p_occ, d_occ):
    size = np.zeros(len(p_occ))
    for i in range(len(p_occ)):
        total = s_occ[i] + p_occ[i] + d_occ[i]
        if total > 1e-6:
            size[i] = 2 * min(p_occ[i] / total, d_occ[i] / total)
    return size


def extract_orbital_weights(band):
    s_occ = band[:, 2]
    p_occ = band[:, 3] + band[:, 4] + band[:, 5]
    d_occ = band[:, 6] + band[:, 7] + band[:, 8] + band[:, 9] + band[:, 10]
    return s_occ, p_occ, d_occ


def draw_panel(ax, file_up, file_dw, sse_band_index, sse_value,
               klabels, kpositions, energy_range,
               s_normal, s_sse, show_colorbar, show_ylabel):
    """
    밴드 하나를 ax에 그림.
    colorbar 공간은 show_colorbar 여부에 관계없이 항상 예약하여
    밴드 영역 크기가 두 패널에서 동일하게 유지됨.
    """
    bands_up = read_pband_data(file_up)
    bands_dw = read_pband_data(file_dw)

    n_bands  = len(bands_up)
    n_kpts   = len(bands_up[0])
    max_kpath = max(b[:, 0].max() for b in bands_up)
    sse_band_0 = sse_band_index - 1   # 0-based

    print(f"  Bands: {n_bands}, K-points: {n_kpts}, "
          f"SSE band: {sse_band_index} (1-based)")

    scatter_up = scatter_dw = None

    # ── Spin-up ──
    for idx, band in enumerate(bands_up):
        kpath  = band[:, 0]
        energy = band[:, 1]
        s_occ, p_occ, d_occ = extract_orbital_weights(band)
        cv   = calculate_hyb(s_occ, p_occ, d_occ)
        mask = cv > 0
        is_sse = (idx == sse_band_0)
        sc = ax.scatter(
            kpath[mask], energy[mask],
            s          = s_sse if is_sse else s_normal,
            c          = cv[mask],
            cmap       = 'Reds', vmin=0., vmax=1.0,
            alpha      = 1,
            edgecolors = 'darkred' if is_sse else 'none',
            linewidths = 0.3 if is_sse else 0,
            zorder     = 5 if is_sse else 2,
        ) if np.any(mask) else None
        if sc is not None:
            scatter_up = sc

    # ── Spin-down ──
    for idx, band in enumerate(bands_dw):
        kpath  = band[:, 0]
        energy = band[:, 1]
        s_occ, p_occ, d_occ = extract_orbital_weights(band)
        cv   = calculate_hyb(s_occ, p_occ, d_occ)
        mask = cv > 0
        is_sse = (idx == sse_band_0)
        sc = ax.scatter(
            kpath[mask], energy[mask],
            s          = s_sse if is_sse else s_normal,
            c          = cv[mask],
            cmap       = 'Blues', vmin=0., vmax=1.0,
            alpha      = 1,
            edgecolors = 'darkblue' if is_sse else 'none',
            linewidths = 0.3 if is_sse else 0,
            zorder     = 5 if is_sse else 2,
        ) if np.any(mask) else None
        if sc is not None:
            scatter_dw = sc

    # ── 축 ──
    y_min, y_max = energy_range
    ax.set_xlim(0, max_kpath)
    ax.set_ylim(energy_range)
    ax.set_yticks(np.arange(y_min, y_max + 1, 1))
    ax.tick_params(axis='both', which='major', labelsize=25)

    if show_ylabel:
        ax.set_ylabel(r'$\mathrm{E - E_F\ (eV)}$', fontsize=25)
    # ytick 숫자는 show_ylabel 여부와 무관하게 항상 표시

    # K-path 라벨
    kpos = (np.linspace(0, max_kpath, len(klabels))
            if kpositions == 'auto' else np.asarray(kpositions))
    ax.set_xticks(kpos)
    ax.set_xticklabels(klabels, fontsize=25)

    for kp in kpos[1:-1]:
        ax.axvline(x=kp, color='black', lw=1.0, linestyle='--', dashes=(4, 3), zorder=1)
    ax.axhline(y=0, color='black', ls='--', lw=1.2, zorder=1)

    # ── 컬러바 공간 — 항상 예약, visible 여부만 다름 ──
    # → 두 패널의 밴드 영역 크기가 동일하게 유지됨
    divider = make_axes_locatable(ax)
    cax1 = divider.append_axes("right", size="4%", pad=0.05)
    cax2 = divider.append_axes("right", size="4%", pad=0.9)

    if show_colorbar:
        if scatter_up is not None:
            cbar1 = plt.colorbar(scatter_up, cax=cax1)
            cbar1.set_label('spin-up',   fontsize=25, rotation=90, labelpad=1)
            cbar1.ax.tick_params(labelsize=20)
        if scatter_dw is not None:
            cbar2 = plt.colorbar(scatter_dw, cax=cax2)
            cbar2.set_label('spin-down', fontsize=25, rotation=90, labelpad=1)
            cbar2.ax.tick_params(labelsize=20)
    else:
        cax1.set_visible(False)
        cax2.set_visible(False)


# ============================================================================
# 실행
# ============================================================================

if __name__ == '__main__':
    n = len(PANELS)
    fig, axes = plt.subplots(1, n, figsize=(5.5 * n, 6))
    if n == 1:
        axes = [axes]

    for ax, cfg in zip(axes, PANELS):
        print(f"\nDrawing: {cfg['file_up']}")
        draw_panel(
            ax            = ax,
            file_up       = cfg['file_up'],
            file_dw       = cfg['file_dw'],
            sse_band_index= cfg['sse_band_index'],
            sse_value     = cfg['sse_value'],
            klabels       = cfg['klabels'],
            kpositions    = cfg['kpositions'],
            energy_range  = cfg['energy_range'],
            s_normal      = cfg['s_normal'],
            s_sse         = cfg['s_sse'],
            show_colorbar = cfg['show_colorbar'],
            show_ylabel   = cfg['show_ylabel'],
        )

    plt.tight_layout()

    if SAVE_PATH:
        plt.savefig(SAVE_PATH, dpi=DPI, bbox_inches='tight')
        print(f"\nSaved: {SAVE_PATH}")

    plt.show()

In [ ]:
#!/usr/bin/env python3
"""
Spin-resolved pd-hybridization fatband plotter — combined two-panel figure.

Left panel  : VO   (no colorbar, with y-label)
Right panel : CrSb (with colorbar, no y-label)

Band-area sizes are kept identical by always reserving colorbar space
and toggling visibility.

Usage:
    python plot_fatband_combined.py
"""

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable
plt.rcParams['mathtext.fontset'] = 'cm'
plt.rcParams['font.family'] = 'Times New Roman'


# ============================================================================
# CONFIG — 여기만 수정하세요
# ============================================================================

PANELS = [
    # ── 왼쪽 패널: VO ──────────────────────────────────────────────────────
    dict(
        file_up        = '../final_structure/CoS/PBAND_SUM_UP.dat',
        file_dw        = '../final_structure/CoS/PBAND_SUM_DW.dat',
        sse_band_index = 14,       # 1-based
        sse_kpt_index  = 19,       # 1-based (참고용)
        sse_value      = 1.103211,    # eV
        klabels        = [r'L', r'$\Gamma$', r"L'"],
        kpositions     = [0, 0.562, 1.125],
        energy_range   = (-4, 4),
        s_normal       = 20,
        s_sse          = 80,
        show_colorbar  = False,    # 왼쪽: colorbar 없음
        show_ylabel    = True,     # 왼쪽: y-label 있음
    ),
    # ── 오른쪽 패널: CrSb ──────────────────────────────────────────────────
    dict(
        file_up        = '../final_structure/FeAs/PBAND_SUM_UP.dat',
        file_dw        = '../final_structure/FeAs/PBAND_SUM_DW.dat',
        sse_band_index = 12,
        sse_kpt_index  = 18,
        sse_value      = 1.088758,
        klabels        = [r'L', r'$\Gamma$', r"L'"],
        kpositions     = [0, 1.107, 2.217],
        energy_range   = (-4, 4),
        s_normal       = 20,
        s_sse          = 80,
        show_colorbar  = True,     # 오른쪽: colorbar 있음
        show_ylabel    = False,    # 오른쪽: y-label 없음
    ),
]

# 저장 설정
SAVE_PATH = 'SI_fatband_combined_CoS_FeAs.png'   # None이면 저장 안 함
DPI       = 300

# 패널 크기 (인치)
PANEL_W, PANEL_H = 6.4, 6.4


# ============================================================================
# 함수 정의 — 수정 불필요
# ============================================================================

def read_pband_data(filename):
    bands, current_band = [], []
    with open(filename, 'r') as f:
        for line in f:
            if line.startswith('#'):
                continue
            if line.strip() == '':
                if current_band:
                    bands.append(np.array(current_band))
                    current_band = []
            else:
                current_band.append(list(map(float, line.split())))
    if current_band:
        bands.append(np.array(current_band))
    return bands


def calculate_hyb(s_occ, p_occ, d_occ):
    size = np.zeros(len(p_occ))
    for i in range(len(p_occ)):
        total = s_occ[i] + p_occ[i] + d_occ[i]
        if total > 1e-6:
            size[i] = 2 * min(p_occ[i] / total, d_occ[i] / total)
    return size


def extract_orbital_weights(band):
    s_occ = band[:, 2]
    p_occ = band[:, 3] + band[:, 4] + band[:, 5]
    d_occ = band[:, 6] + band[:, 7] + band[:, 8] + band[:, 9] + band[:, 10]
    return s_occ, p_occ, d_occ


def draw_panel(ax, file_up, file_dw, sse_band_index, sse_value,
               klabels, kpositions, energy_range,
               s_normal, s_sse, show_colorbar, show_ylabel):
    """
    밴드 하나를 ax에 그림.
    colorbar 공간은 show_colorbar 여부에 관계없이 항상 예약하여
    밴드 영역 크기가 두 패널에서 동일하게 유지됨.
    """
    bands_up = read_pband_data(file_up)
    bands_dw = read_pband_data(file_dw)

    n_bands  = len(bands_up)
    n_kpts   = len(bands_up[0])
    max_kpath = max(b[:, 0].max() for b in bands_up)
    sse_band_0 = sse_band_index - 1   # 0-based

    print(f"  Bands: {n_bands}, K-points: {n_kpts}, "
          f"SSE band: {sse_band_index} (1-based)")

    scatter_up = scatter_dw = None

    # ── Spin-up ──
    for idx, band in enumerate(bands_up):
        kpath  = band[:, 0]
        energy = band[:, 1]
        s_occ, p_occ, d_occ = extract_orbital_weights(band)
        cv   = calculate_hyb(s_occ, p_occ, d_occ)
        mask = cv > 0
        is_sse = (idx == sse_band_0)
        sc = ax.scatter(
            kpath[mask], energy[mask],
            s          = s_sse if is_sse else s_normal,
            c          = cv[mask],
            cmap       = 'Reds', vmin=0., vmax=1.0,
            alpha      = 1,
            edgecolors = 'darkred' if is_sse else 'none',
            linewidths = 0.3 if is_sse else 0,
            zorder     = 5 if is_sse else 2,
        ) if np.any(mask) else None
        if sc is not None:
            scatter_up = sc

    # ── Spin-down ──
    for idx, band in enumerate(bands_dw):
        kpath  = band[:, 0]
        energy = band[:, 1]
        s_occ, p_occ, d_occ = extract_orbital_weights(band)
        cv   = calculate_hyb(s_occ, p_occ, d_occ)
        mask = cv > 0
        is_sse = (idx == sse_band_0)
        sc = ax.scatter(
            kpath[mask], energy[mask],
            s          = s_sse if is_sse else s_normal,
            c          = cv[mask],
            cmap       = 'Blues', vmin=0., vmax=1.0,
            alpha      = 1,
            edgecolors = 'darkblue' if is_sse else 'none',
            linewidths = 0.3 if is_sse else 0,
            zorder     = 5 if is_sse else 2,
        ) if np.any(mask) else None
        if sc is not None:
            scatter_dw = sc

    # ── 축 ──
    y_min, y_max = energy_range
    ax.set_xlim(0, max_kpath)
    ax.set_ylim(energy_range)
    ax.set_yticks(np.arange(y_min, y_max + 1, 1))
    ax.tick_params(axis='both', which='major', labelsize=25)

    if show_ylabel:
        ax.set_ylabel(r'$\mathrm{E - E_F\ (eV)}$', fontsize=25)
    # ytick 숫자는 show_ylabel 여부와 무관하게 항상 표시

    # K-path 라벨
    kpos = (np.linspace(0, max_kpath, len(klabels))
            if kpositions == 'auto' else np.asarray(kpositions))
    ax.set_xticks(kpos)
    ax.set_xticklabels(klabels, fontsize=25)

    for kp in kpos[1:-1]:
        ax.axvline(x=kp, color='black', lw=1.0, linestyle='--', dashes=(4, 3), zorder=1)
    ax.axhline(y=0, color='black', ls='--', lw=1.2, zorder=1)

    # ── 컬러바 공간 — 항상 예약, visible 여부만 다름 ──
    # → 두 패널의 밴드 영역 크기가 동일하게 유지됨
    divider = make_axes_locatable(ax)
    cax1 = divider.append_axes("right", size="4%", pad=0.05)
    cax2 = divider.append_axes("right", size="4%", pad=0.9)

    if show_colorbar:
        if scatter_up is not None:
            cbar1 = plt.colorbar(scatter_up, cax=cax1)
            cbar1.set_label('spin-up',   fontsize=25, rotation=90, labelpad=1)
            cbar1.ax.tick_params(labelsize=20)
        if scatter_dw is not None:
            cbar2 = plt.colorbar(scatter_dw, cax=cax2)
            cbar2.set_label('spin-down', fontsize=25, rotation=90, labelpad=1)
            cbar2.ax.tick_params(labelsize=20)
    else:
        cax1.set_visible(False)
        cax2.set_visible(False)


# ============================================================================
# 실행
# ============================================================================

if __name__ == '__main__':
    n = len(PANELS)
    fig, axes = plt.subplots(1, n, figsize=(5.5 * n, 6))
    if n == 1:
        axes = [axes]

    for ax, cfg in zip(axes, PANELS):
        print(f"\nDrawing: {cfg['file_up']}")
        draw_panel(
            ax            = ax,
            file_up       = cfg['file_up'],
            file_dw       = cfg['file_dw'],
            sse_band_index= cfg['sse_band_index'],
            sse_value     = cfg['sse_value'],
            klabels       = cfg['klabels'],
            kpositions    = cfg['kpositions'],
            energy_range  = cfg['energy_range'],
            s_normal      = cfg['s_normal'],
            s_sse         = cfg['s_sse'],
            show_colorbar = cfg['show_colorbar'],
            show_ylabel   = cfg['show_ylabel'],
        )

    plt.tight_layout()

    if SAVE_PATH:
        plt.savefig(SAVE_PATH, dpi=DPI, bbox_inches='tight')
        print(f"\nSaved: {SAVE_PATH}")

    plt.show()

In [ ]:
#!/usr/bin/env python3
"""
Spin-resolved pd-hybridization fatband plotter — combined two-panel figure.

Left panel  : VO   (no colorbar, with y-label)
Right panel : CrSb (with colorbar, no y-label)

Band-area sizes are kept identical by always reserving colorbar space
and toggling visibility.

Usage:
    python plot_fatband_combined.py
"""

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable
plt.rcParams['mathtext.fontset'] = 'cm'
plt.rcParams['font.family'] = 'Times New Roman'


# ============================================================================
# CONFIG — 여기만 수정하세요
# ============================================================================

PANELS = [
    # ── 왼쪽 패널: VO ──────────────────────────────────────────────────────
    dict(
        file_up        = '../final_structure/CoO/PBAND_SUM_UP.dat',
        file_dw        = '../final_structure/CoO/PBAND_SUM_DW.dat',
        sse_band_index = 15,       # 1-based
        sse_kpt_index  = 1,       # 1-based (참고용)
        sse_value      = 0.972368,    # eV
        klabels        = [r'T$_2$', r'$\Gamma$', r"U$_2$"],
        kpositions     = [0, 1.457, 2.655],
        energy_range   = (-4, 4),
        s_normal       = 20,
        s_sse          = 80,
        show_colorbar  = False,    # 왼쪽: colorbar 없음
        show_ylabel    = True,     # 왼쪽: y-label 있음
    ),
    # ── 오른쪽 패널: CrSb ──────────────────────────────────────────────────
    dict(
        file_up        = '../final_structure/CrS/PBAND_SUM_UP.dat',
        file_dw        = '../final_structure/CrS/PBAND_SUM_DW.dat',
        sse_band_index = 18,
        sse_kpt_index  = 12,
        sse_value      = 0.549354,
        klabels        = [r'T$_2$', r'$\Gamma$', r"U$_2$"],
        kpositions     = [0, 1.267, 2.227],
        energy_range   = (-4, 4),
        s_normal       = 20,
        s_sse          = 80,
        show_colorbar  = True,     # 오른쪽: colorbar 있음
        show_ylabel    = False,    # 오른쪽: y-label 없음
    ),
]

# 저장 설정
SAVE_PATH = 'SI_fatband_combined_CoO_CrS.png'   # None이면 저장 안 함
DPI       = 300

# 패널 크기 (인치)
PANEL_W, PANEL_H = 6.4, 6.4


# ============================================================================
# 함수 정의 — 수정 불필요
# ============================================================================

def read_pband_data(filename):
    bands, current_band = [], []
    with open(filename, 'r') as f:
        for line in f:
            if line.startswith('#'):
                continue
            if line.strip() == '':
                if current_band:
                    bands.append(np.array(current_band))
                    current_band = []
            else:
                current_band.append(list(map(float, line.split())))
    if current_band:
        bands.append(np.array(current_band))
    return bands


def calculate_hyb(s_occ, p_occ, d_occ):
    size = np.zeros(len(p_occ))
    for i in range(len(p_occ)):
        total = s_occ[i] + p_occ[i] + d_occ[i]
        if total > 1e-6:
            size[i] = 2 * min(p_occ[i] / total, d_occ[i] / total)
    return size


def extract_orbital_weights(band):
    s_occ = band[:, 2]
    p_occ = band[:, 3] + band[:, 4] + band[:, 5]
    d_occ = band[:, 6] + band[:, 7] + band[:, 8] + band[:, 9] + band[:, 10]
    return s_occ, p_occ, d_occ


def draw_panel(ax, file_up, file_dw, sse_band_index, sse_value,
               klabels, kpositions, energy_range,
               s_normal, s_sse, show_colorbar, show_ylabel):
    """
    밴드 하나를 ax에 그림.
    colorbar 공간은 show_colorbar 여부에 관계없이 항상 예약하여
    밴드 영역 크기가 두 패널에서 동일하게 유지됨.
    """
    bands_up = read_pband_data(file_up)
    bands_dw = read_pband_data(file_dw)

    n_bands  = len(bands_up)
    n_kpts   = len(bands_up[0])
    max_kpath = max(b[:, 0].max() for b in bands_up)
    sse_band_0 = sse_band_index - 1   # 0-based

    print(f"  Bands: {n_bands}, K-points: {n_kpts}, "
          f"SSE band: {sse_band_index} (1-based)")

    scatter_up = scatter_dw = None

    # ── Spin-up ──
    for idx, band in enumerate(bands_up):
        kpath  = band[:, 0]
        energy = band[:, 1]
        s_occ, p_occ, d_occ = extract_orbital_weights(band)
        cv   = calculate_hyb(s_occ, p_occ, d_occ)
        mask = cv > 0
        is_sse = (idx == sse_band_0)
        sc = ax.scatter(
            kpath[mask], energy[mask],
            s          = s_sse if is_sse else s_normal,
            c          = cv[mask],
            cmap       = 'Reds', vmin=0., vmax=1.0,
            alpha      = 1,
            edgecolors = 'darkred' if is_sse else 'none',
            linewidths = 0.3 if is_sse else 0,
            zorder     = 5 if is_sse else 2,
        ) if np.any(mask) else None
        if sc is not None:
            scatter_up = sc

    # ── Spin-down ──
    for idx, band in enumerate(bands_dw):
        kpath  = band[:, 0]
        energy = band[:, 1]
        s_occ, p_occ, d_occ = extract_orbital_weights(band)
        cv   = calculate_hyb(s_occ, p_occ, d_occ)
        mask = cv > 0
        is_sse = (idx == sse_band_0)
        sc = ax.scatter(
            kpath[mask], energy[mask],
            s          = s_sse if is_sse else s_normal,
            c          = cv[mask],
            cmap       = 'Blues', vmin=0., vmax=1.0,
            alpha      = 1,
            edgecolors = 'darkblue' if is_sse else 'none',
            linewidths = 0.3 if is_sse else 0,
            zorder     = 5 if is_sse else 2,
        ) if np.any(mask) else None
        if sc is not None:
            scatter_dw = sc

    # ── 축 ──
    y_min, y_max = energy_range
    ax.set_xlim(0, max_kpath)
    ax.set_ylim(energy_range)
    ax.set_yticks(np.arange(y_min, y_max + 1, 1))
    ax.tick_params(axis='both', which='major', labelsize=25)

    if show_ylabel:
        ax.set_ylabel(r'$\mathrm{E - E_F\ (eV)}$', fontsize=25)
    # ytick 숫자는 show_ylabel 여부와 무관하게 항상 표시

    # K-path 라벨
    kpos = (np.linspace(0, max_kpath, len(klabels))
            if kpositions == 'auto' else np.asarray(kpositions))
    ax.set_xticks(kpos)
    ax.set_xticklabels(klabels, fontsize=25)

    for kp in kpos[1:-1]:
        ax.axvline(x=kp, color='black', lw=1.0, linestyle='--', dashes=(4, 3), zorder=1)
    ax.axhline(y=0, color='black', ls='--', lw=1.2, zorder=1)

    # ── 컬러바 공간 — 항상 예약, visible 여부만 다름 ──
    # → 두 패널의 밴드 영역 크기가 동일하게 유지됨
    divider = make_axes_locatable(ax)
    cax1 = divider.append_axes("right", size="4%", pad=0.05)
    cax2 = divider.append_axes("right", size="4%", pad=0.9)

    if show_colorbar:
        if scatter_up is not None:
            cbar1 = plt.colorbar(scatter_up, cax=cax1)
            cbar1.set_label('spin-up',   fontsize=25, rotation=90, labelpad=1)
            cbar1.ax.tick_params(labelsize=20)
        if scatter_dw is not None:
            cbar2 = plt.colorbar(scatter_dw, cax=cax2)
            cbar2.set_label('spin-down', fontsize=25, rotation=90, labelpad=1)
            cbar2.ax.tick_params(labelsize=20)
    else:
        cax1.set_visible(False)
        cax2.set_visible(False)


# ============================================================================
# 실행
# ============================================================================

if __name__ == '__main__':
    n = len(PANELS)
    fig, axes = plt.subplots(1, n, figsize=(5.5 * n, 6))
    if n == 1:
        axes = [axes]

    for ax, cfg in zip(axes, PANELS):
        print(f"\nDrawing: {cfg['file_up']}")
        draw_panel(
            ax            = ax,
            file_up       = cfg['file_up'],
            file_dw       = cfg['file_dw'],
            sse_band_index= cfg['sse_band_index'],
            sse_value     = cfg['sse_value'],
            klabels       = cfg['klabels'],
            kpositions    = cfg['kpositions'],
            energy_range  = cfg['energy_range'],
            s_normal      = cfg['s_normal'],
            s_sse         = cfg['s_sse'],
            show_colorbar = cfg['show_colorbar'],
            show_ylabel   = cfg['show_ylabel'],
        )

    plt.tight_layout()

    if SAVE_PATH:
        plt.savefig(SAVE_PATH, dpi=DPI, bbox_inches='tight')
        print(f"\nSaved: {SAVE_PATH}")

    plt.show()

In [ ]:
# ============================================================================
# CONFIG — 여기만 수정하세요
# ============================================================================

# 1. 파일 경로
FILE_UP = '../final_structure/NiS/PBAND_SUM_UP.dat'
FILE_DW = '../final_structure/NiS/PBAND_SUM_DW.dat'

# 2. SSE 정보 (SSE 스크립트 출력에서 가져오세요)
#    Band Index는 1-based (SSE 출력 그대로), 코드가 자동으로 0-based 변환
SSE_BAND_INDEX = 16       # Band Index from SSE output
SSE_KPOINT_INDEX = 63     # K-Point Index from SSE output (1-based)
SSE_VALUE = 0.8229          # Max Energy Difference (eV)

# 3. K-path 라벨 설정
#    k-point 경로의 고대칭점 라벨
#    labels: 표시할 라벨 리스트
#    positions: 'auto' 이면 [시작, 중간, 끝] 자동 배치
#               직접 kpath 값 리스트로 지정 가능 (e.g., [0, 0.5, 1.0, 1.5])
KLABELS = [r"L", r"$\Gamma$", r"L'"]
KPOSITIONS = 'auto'  # 'auto' or list of kpath values

# 4. 에너지 범위
ENERGY_RANGE = (-4, 4)

# 5. 마커 크기
S_NORMAL = 20    # 일반 밴드 마커 크기
S_SSE = 80       # SSE 밴드 마커 크기

# 6. 저장 설정
SAVE_PATH = 'NiS_fatband.png'  # None이면 저장 안 함
DPI = 300

# ============================================================================
# 실행
# ============================================================================
if __name__ == '__main__':
    fig, ax = plot_fatband(
        file_up=FILE_UP,
        file_dw=FILE_DW,
        sse_band_1based=SSE_BAND_INDEX,
        sse_value=SSE_VALUE,
        klabels=KLABELS,
        kpositions=KPOSITIONS,
        energy_range=ENERGY_RANGE,
        s_normal=S_NORMAL,
        s_sse=S_SSE,
        save_path=SAVE_PATH,
        dpi=DPI,
    )

In [ ]:
# ============================================================================
# CONFIG — 여기만 수정하세요
# ============================================================================

# 1. 파일 경로
FILE_UP = '../final_structure/FeS/PBAND_SUM_UP.dat'
FILE_DW = '../final_structure/FeS/PBAND_SUM_DW.dat'

# 2. SSE 정보 (SSE 스크립트 출력에서 가져오세요)
#    Band Index는 1-based (SSE 출력 그대로), 코드가 자동으로 0-based 변환
SSE_BAND_INDEX = 14       # Band Index from SSE output
SSE_KPOINT_INDEX = 66     # K-Point Index from SSE output (1-based)
SSE_VALUE = 1.2966          # Max Energy Difference (eV)

# 3. K-path 라벨 설정
#    k-point 경로의 고대칭점 라벨
#    labels: 표시할 라벨 리스트
#    positions: 'auto' 이면 [시작, 중간, 끝] 자동 배치
#               직접 kpath 값 리스트로 지정 가능 (e.g., [0, 0.5, 1.0, 1.5])
KLABELS = [r"T$_{2}$", r"$\Gamma$", r"U$_{2}$"]
KPOSITIONS = [0, 1.289, 2.579]  # 'auto' or list of kpath values

# 4. 에너지 범위
ENERGY_RANGE = (-4, 4)

# 5. 마커 크기
S_NORMAL = 20    # 일반 밴드 마커 크기
S_SSE = 80       # SSE 밴드 마커 크기

# 6. 저장 설정
SAVE_PATH = 'FeS_fatband.png'  # None이면 저장 안 함
DPI = 300

# ============================================================================
# 실행
# ============================================================================
if __name__ == '__main__':
    fig, ax = plot_fatband(
        file_up=FILE_UP,
        file_dw=FILE_DW,
        sse_band_1based=SSE_BAND_INDEX,
        sse_value=SSE_VALUE,
        klabels=KLABELS,
        kpositions=KPOSITIONS,
        energy_range=ENERGY_RANGE,
        s_normal=S_NORMAL,
        s_sse=S_SSE,
        save_path=SAVE_PATH,
        dpi=DPI,
    )

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.collections import PathCollection
from mpl_toolkits.axes_grid1 import make_axes_locatable

def read_pband_data(filename):
    """PBAND 데이터를 읽어들이는 함수"""
    bands = []
    current_band = []
    
    with open(filename, 'r') as f:
        for line in f:
            if line.startswith('#'):
                continue
            if line.strip() == '':
                if current_band:
                    bands.append(np.array(current_band))
                    current_band = []
            else:
                data = list(map(float, line.split()))
                current_band.append(data)
    
    if current_band:
        bands.append(np.array(current_band))
    
    return bands

def calculate_marker_size(s_occ, p_occ, d_occ):
    """
    색깔 강도를 계산하는 함수 (Minimum 방식)
    size = 2 * min(p_global, d_global)
    where p_global = p/(s+p+d), d_global = d/(s+p+d)
    """
    size = np.zeros(len(p_occ))
    
    for i in range(len(p_occ)):
        # 1. '전체' 기여도 계산 (s 오비탈 포함)
        total_occ = s_occ[i] + p_occ[i] + d_occ[i]
        
        if total_occ > 1e-6:  # 0으로 나누기 방지
            # 2. '전체' 대비 '글로벌' 비율 계산
            p_global = p_occ[i] / total_occ
            d_global = d_occ[i] / total_occ
            
            # 3. Minimum 방식: 더 작은 값이 혼성화의 한계
            # 4. 2배하여 0~1 스케일로 만듦 (p=d=0.5일 때 1.0)
            size[i] = 2 * min(p_global, d_global)
        else:
            size[i] = 0
    
    return size

def plot_fatband(filename_up, filename_dw=None, energy_range=(-4, 4), save_path=None):
    """
    Fat band 플로팅 함수 - 색깔로 표현
    
    Parameters:
    -----------
    filename_up : str
        Spin-up PBAND 데이터 파일 경로
    filename_dw : str, optional
        Spin-down PBAND 데이터 파일 경로
    energy_range : tuple, optional
        에너지 범위 (min, max), 기본값은 (-4, 4)
    save_path : str, optional
        그림 저장 경로. None이면 저장하지 않음
    
    Returns:
    --------
    fig, ax : matplotlib figure and axes objects
    """
    
    # Font 설정
    plt.rcParams['font.family'] = 'Times New Roman'
    plt.rcParams['mathtext.fontset'] = 'cm'
    
    # 데이터 읽기
    bands_up = read_pband_data(filename_up)
    
    if filename_dw:
        bands_dw = read_pband_data(filename_dw)
    else:
        bands_dw = None
    
    # 플롯 설정
    fig, ax = plt.subplots(figsize=(6, 4))
    
    # 데이터의 최대 kpath 값 찾기
    max_kpath = 0
    for band in bands_up:
        max_kpath = max(max_kpath, band[:, 0].max())
    if bands_dw:
        for band in bands_dw:
            max_kpath = max(max_kpath, band[:, 0].max())
    
    # Spin-up 밴드 그리기
    for band in bands_up:
        kpath = band[:, 0]
        energy = band[:, 1]
        
        # s 오비탈 (column 3 -> index 2)
        s_occ = band[:, 2]
        
        # p 궤도 합: py + pz + px (columns 4, 5, 6 -> indices 3, 4, 5)
        p_occ = band[:, 3] + band[:, 4] + band[:, 5]
        
        # d 궤도 합: dxy + dyz + dz2 + dxz + dx2-y2 (columns 7-11 -> indices 6-10)
        d_occ = band[:, 6] + band[:, 7] + band[:, 8] + band[:, 9] + band[:, 10]
        
        # 색깔 값 계산
        color_values = calculate_marker_size(s_occ, p_occ, d_occ)
        
        # Fat band 그리기 (색깔로) - 선 없이 원만 그리기
        # 0이 아닌 값만 표시
        mask = color_values > 0
        if np.any(mask):
            scatter = ax.scatter(kpath[mask], energy[mask], s=25, 
                      c=color_values[mask], cmap='Reds', 
                      vmin=0, vmax=1.0, alpha=1, edgecolors='none')
    
    # Spin-down 밴드 그리기 (있는 경우)
    if bands_dw:
        for band in bands_dw:
            kpath = band[:, 0]
            energy = band[:, 1]
            
            # s 오비탈 (column 3 -> index 2)
            s_occ = band[:, 2]
            
            # p 궤도 합: py + pz + px (columns 4, 5, 6 -> indices 3, 4, 5)
            p_occ = band[:, 3] + band[:, 4] + band[:, 5]
            
            # d 궤도 합: dxy + dyz + dz2 + dxz + dx2-y2 (columns 7-11 -> indices 6-10)
            d_occ = band[:, 6] + band[:, 7] + band[:, 8] + band[:, 9] + band[:, 10]
            
            color_values = calculate_marker_size(s_occ, p_occ, d_occ)
            
            # Fat band 그리기 (색깔로) - 선 없이 원만 그리기
            mask = color_values > 0
            if np.any(mask):
                scatter2 = ax.scatter(kpath[mask], energy[mask], s=25, 
                          c=color_values[mask], cmap='Blues', 
                          vmin=0, vmax=1.0, alpha=1, edgecolors='none')
    
    # 축 설정
    ax.set_xlabel('', fontsize=20)  # K-Path 레이블 제거
    ax.set_ylabel('Energy (eV)', fontsize=20)
    # 제목 제거
    
    # x축 범위 및 tick 설정
    ax.set_xlim(0, max_kpath)
    mid_kpath = max_kpath / 2
    ax.set_xticks([0, mid_kpath, max_kpath])
    ax.set_xticklabels(['', r'$\Gamma$', ''])
    ax.set_yticks([-4,-3,-2,-1,0,1,2,3,4])
    ax.set_yticklabels([-4,-3,-2,-1,0,1,2,3,4])
    
    # 에너지 범위 설정
    if energy_range:
        ax.set_ylim(energy_range)
    
    # Tick 크기 설정
    ax.tick_params(axis='both', labelsize=15)
    
    # 0 에너지 선 (Fermi level) - 검정색으로 진하게
    ax.axhline(y=0, color='black', linestyle='--', linewidth=1, alpha=1.0)
    
    # 컬러바 추가
    if bands_dw:
        # 두 개의 컬러바 추가
        divider = make_axes_locatable(ax)
        cax1 = divider.append_axes("right", size="2%", pad=0.05)
        cax2 = divider.append_axes("right", size="2%", pad=0.9)
        
        # Spin-up 컬러바
        cbar1 = plt.colorbar(scatter, cax=cax1)
        cbar1.set_label('spin-up', fontsize=20, rotation=90, labelpad=9)
        cbar1.ax.tick_params(labelsize=15)
        
        # Spin-down 컬러바
        cbar2 = plt.colorbar(scatter2, cax=cax2)
        cbar2.set_label('spin-down', fontsize=20, rotation=90, labelpad=9)
        cbar2.ax.tick_params(labelsize=15)
    else:
        cbar = plt.colorbar(scatter, ax=ax, label='Value')
        cbar.ax.tick_params(labelsize=15)
    
    plt.tight_layout()
    
    # 저장 (경로가 지정된 경우)
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Figure saved to {save_path}")
    
    return fig, ax


# ============================================================================
# 사용 예시 (Jupyter Notebook에서 실행)
# ============================================================================

# 1. 파일 경로 설정 (여기만 수정하세요!)
file_up = 'CrSb/PBAND_SUM_UP.dat'  # Spin-up 파일 경로
file_dw = 'CrSb/PBAND_SUM_DW.dat'  # Spin-down 파일 경로

# 2. Fat band 그리기
fig, ax = plot_fatband(
    filename_up=file_up,
    filename_dw=file_dw,
    energy_range=(-4, 4),  # 에너지 범위 (eV)
    save_path='CrSb_pd_fatband.png'  # 저장할 파일명 (None이면 저장 안 함)
)

plt.show()

In [ ]:
# 1. 파일 경로 설정 (여기만 수정하세요!)
file_up = 'VO/PBAND_SUM_UP.dat'  # Spin-up 파일 경로
file_dw = 'VO/PBAND_SUM_DW.dat'  # Spin-down 파일 경로

# 2. Fat band 그리기
fig, ax = plot_fatband(
    filename_up=file_up,
    filename_dw=file_dw,
    energy_range=(-4, 4),  # 에너지 범위 (eV)
    save_path='VO_pd_fatband.png'  # 저장할 파일명 (None이면 저장 안 함)
)

plt.show()

In [ ]:
# 1. 파일 경로 설정 (여기만 수정하세요!)
file_up = '../final_structure/NiS/PBAND_SUM_UP.dat'  # Spin-up 파일 경로
file_dw = '../final_structure/NiS/PBAND_SUM_DW.dat'  # Spin-down 파일 경로

# 2. Fat band 그리기
fig, ax = plot_fatband(
    filename_up=file_up,
    filename_dw=file_dw,
    energy_range=(-3, 3),  # 에너지 범위 (eV)
    save_path='NiS_pd_fatband.png'  # 저장할 파일명 (None이면 저장 안 함)
)

plt.show()

In [ ]:
# 1. 파일 경로 설정 (여기만 수정하세요!)
file_up = '../final_structure/NiS/u0/PBAND_SUM_UP.dat'  # Spin-up 파일 경로
file_dw = '../final_structure/NiS/u0/PBAND_SUM_DW.dat'  # Spin-down 파일 경로

# 2. Fat band 그리기
fig, ax = plot_fatband(
    filename_up=file_up,
    filename_dw=file_dw,
    energy_range=(-3, 3),  # 에너지 범위 (eV)
    save_path='NiS_u0_pd_fatband.png'  # 저장할 파일명 (None이면 저장 안 함)
)

plt.show()

In [ ]:
# 1. 파일 경로 설정 (여기만 수정하세요!)
file_up = '../final_structure/NiS/u3/PBAND_SUM_UP.dat'  # Spin-up 파일 경로
file_dw = '../final_structure/NiS/u3/PBAND_SUM_DW.dat'  # Spin-down 파일 경로

# 2. Fat band 그리기
fig, ax = plot_fatband(
    filename_up=file_up,
    filename_dw=file_dw,
    energy_range=(-3, 3),  # 에너지 범위 (eV)
    save_path='NiS_u3_pd_fatband.png'  # 저장할 파일명 (None이면 저장 안 함)
)

plt.show()

In [ ]:
# 1. 파일 경로 설정 (여기만 수정하세요!)
file_up = '../final_structure/CoS/PBAND_SUM_UP.dat'  # Spin-up 파일 경로
file_dw = '../final_structure/CoS/PBAND_SUM_DW.dat'  # Spin-down 파일 경로

# 2. Fat band 그리기
fig, ax = plot_fatband(
    filename_up=file_up,
    filename_dw=file_dw,
    energy_range=(-4, 4),  # 에너지 범위 (eV)
    save_path='CoS_pd_fatband.png'  # 저장할 파일명 (None이면 저장 안 함)
)

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.collections import PathCollection
from mpl_toolkits.axes_grid1 import make_axes_locatable

def read_pband_data(filename):
    """PBAND 데이터를 읽어들이는 함수"""
    bands = []
    current_band = []
    
    with open(filename, 'r') as f:
        for line in f:
            if line.startswith('#'):
                continue
            if line.strip() == '':
                if current_band:
                    bands.append(np.array(current_band))
                    current_band = []
            else:
                data = list(map(float, line.split()))
                current_band.append(data)
    
    if current_band:
        bands.append(np.array(current_band))
    
    return bands

def calculate_marker_size(s_occ, p_occ, d_occ):
    """
    색깔 강도를 계산하는 함수 (Minimum 방식)
    size = 2 * min(p_global, d_global)
    where p_global = p/(s+p+d), d_global = d/(s+p+d)
    """
    size = np.zeros(len(p_occ))
    
    for i in range(len(p_occ)):
        # 1. '전체' 기여도 계산 (s 오비탈 포함)
        total_occ = s_occ[i] + p_occ[i] + d_occ[i]
        
        if total_occ > 1e-6:  # 0으로 나누기 방지
            # 2. '전체' 대비 '글로벌' 비율 계산
            p_global = p_occ[i] / total_occ
            d_global = d_occ[i] / total_occ
            
            # 3. Minimum 방식: 더 작은 값이 혼성화의 한계
            # 4. 2배하여 0~1 스케일로 만듦 (p=d=0.5일 때 1.0)
            size[i] = 2 * min(p_global, d_global)
        else:
            size[i] = 0
    
    return size

def plot_fatband(filename_up, filename_dw=None, energy_range=(-4, 4), save_path=None):
    """
    Fat band 플로팅 함수 - 색깔로 표현
    
    Parameters:
    -----------
    filename_up : str
        Spin-up PBAND 데이터 파일 경로
    filename_dw : str, optional
        Spin-down PBAND 데이터 파일 경로
    energy_range : tuple, optional
        에너지 범위 (min, max), 기본값은 (-4, 4)
    save_path : str, optional
        그림 저장 경로. None이면 저장하지 않음
    
    Returns:
    --------
    fig, ax : matplotlib figure and axes objects
    """
    
    # Font 설정
    plt.rcParams['font.family'] = 'Times New Roman'
    plt.rcParams['mathtext.fontset'] = 'cm'
    
    # 데이터 읽기
    bands_up = read_pband_data(filename_up)
    
    if filename_dw:
        bands_dw = read_pband_data(filename_dw)
    else:
        bands_dw = None
    
    # 플롯 설정
    fig, ax = plt.subplots(figsize=(8, 4))
    
    # 데이터의 최대 kpath 값 찾기
    max_kpath = 0
    for band in bands_up:
        max_kpath = max(max_kpath, band[:, 0].max())
    if bands_dw:
        for band in bands_dw:
            max_kpath = max(max_kpath, band[:, 0].max())
    
    # Spin-up 밴드 그리기
    for band in bands_up:
        kpath = band[:, 0]
        energy = band[:, 1]
        
        # s 오비탈 (column 3 -> index 2)
        s_occ = band[:, 2]
        
        # p 궤도 합: py + pz + px (columns 4, 5, 6 -> indices 3, 4, 5)
        p_occ = band[:, 3] + band[:, 4] + band[:, 5]
        
        # d 궤도 합: dxy + dyz + dz2 + dxz + dx2-y2 (columns 7-11 -> indices 6-10)
        d_occ = band[:, 6] + band[:, 7] + band[:, 8] + band[:, 9] + band[:, 10]
        
        # 색깔 값 계산
        color_values = calculate_marker_size(s_occ, p_occ, d_occ)
        
        # Fat band 그리기 (색깔로) - 선 없이 원만 그리기
        # 0이 아닌 값만 표시
        mask = color_values > 0
        if np.any(mask):
            scatter = ax.scatter(kpath[mask], energy[mask], s=25, 
                      c=color_values[mask], cmap='Reds', 
                      vmin=0, vmax=1.0, alpha=1, edgecolors='none')
    
    # Spin-down 밴드 그리기 (있는 경우)
    if bands_dw:
        for band in bands_dw:
            kpath = band[:, 0]
            energy = band[:, 1]
            
            # s 오비탈 (column 3 -> index 2)
            s_occ = band[:, 2]
            
            # p 궤도 합: py + pz + px (columns 4, 5, 6 -> indices 3, 4, 5)
            p_occ = band[:, 3] + band[:, 4] + band[:, 5]
            
            # d 궤도 합: dxy + dyz + dz2 + dxz + dx2-y2 (columns 7-11 -> indices 6-10)
            d_occ = band[:, 6] + band[:, 7] + band[:, 8] + band[:, 9] + band[:, 10]
            
            color_values = calculate_marker_size(s_occ, p_occ, d_occ)
            
            # Fat band 그리기 (색깔로) - 선 없이 원만 그리기
            mask = color_values > 0
            if np.any(mask):
                scatter2 = ax.scatter(kpath[mask], energy[mask], s=25, 
                          c=color_values[mask], cmap='Blues', 
                          vmin=0, vmax=1.0, alpha=1, edgecolors='none')
    
    # High-symmetry k-points 설정
    kpoint_positions = [0.000, 1.009, 2.016, 2.573, 4.528, 5.817, 7.107, 8.214]
    kpoint_labels = ['Γ', 'X|Y', 'Γ', 'Z|R$_2$', 'Γ', 'T$_2$|U$_2$', 'Γ', 'V$_2$']
    
    # 수직선 그리기 (high-symmetry point 사이 구분)
    for kpos in kpoint_positions[1:-1]:  # 첫점과 끝점 제외
        ax.axvline(x=kpos, color='gray', linestyle='-', linewidth=0.5, alpha=0.5)
    
    # 축 설정
    ax.set_xlabel('', fontsize=20)
    ax.set_ylabel('E - E$_{\mathrm{F}}$ (eV)', fontsize=20)
    
    # x축 범위 및 tick 설정
    ax.set_xlim(kpoint_positions[0], kpoint_positions[-1])
    ax.set_xticks(kpoint_positions)
    ax.set_xticklabels(kpoint_labels)
    
    # y축 tick 설정
    ax.set_yticks([-4,-3,-2,-1,0,1,2,3,4])
    ax.set_yticklabels([-4,-3,-2,-1,0,1,2,3,4])
    
    # 에너지 범위 설정
    if energy_range:
        ax.set_ylim(energy_range)
    
    # Tick 크기 설정
    ax.tick_params(axis='both', labelsize=15)
    
    # 0 에너지 선 (Fermi level) - 검정색으로 진하게
    ax.axhline(y=0, color='black', linestyle='--', linewidth=1, alpha=1.0)
    
    # 컬러바 추가
    if bands_dw:
        # 두 개의 컬러바 추가
        divider = make_axes_locatable(ax)
        cax1 = divider.append_axes("right", size="2%", pad=0.05)
        cax2 = divider.append_axes("right", size="2%", pad=0.9)
        
        # Spin-up 컬러바
        cbar1 = plt.colorbar(scatter, cax=cax1)
        cbar1.set_label('spin-up', fontsize=20, rotation=90, labelpad=9)
        cbar1.ax.tick_params(labelsize=15)
        
        # Spin-down 컬러바
        cbar2 = plt.colorbar(scatter2, cax=cax2)
        cbar2.set_label('spin-down', fontsize=20, rotation=90, labelpad=9)
        cbar2.ax.tick_params(labelsize=15)
    else:
        cbar = plt.colorbar(scatter, ax=ax, label='Value')
        cbar.ax.tick_params(labelsize=15)
    
    plt.tight_layout()
    
    # 저장 (경로가 지정된 경우)
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Figure saved to {save_path}")
    
    return fig, ax

In [ ]:
# 1. 파일 경로 설정 (여기만 수정하세요!)
file_up = '../final_structure/FeS/PBAND_SUM_UP.dat'  # Spin-up 파일 경로
file_dw = '../final_structure/FeS/PBAND_SUM_DW.dat'  # Spin-down 파일 경로

# 2. Fat band 그리기
fig, ax = plot_fatband(
    filename_up=file_up,
    filename_dw=file_dw,
    energy_range=(-4, 4),  # 에너지 범위 (eV)
    save_path='FeS_pd_fatband.png'  # 저장할 파일명 (None이면 저장 안 함)
)

plt.show()